# Building a Knowledge Graph from a Book

This notebook demonstrates Neocortex at realistic scale by indexing the complete *Adventures of Sherlock Holmes* from Project Gutenberg and running multiple analytical queries.

You'll see:
- How to download and prepare a book-length corpus
- Indexing performance at scale (timing and cost)
- Multi-query benchmarking with per-query metrics

In [ ]:
import os
import time
import urllib.request
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from neocortex import GraphRAG, QueryParam
from neocortex._llm import token_tracker

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"
print("Environment ready.")

## Download the Corpus

We'll download *The Adventures of Sherlock Holmes* from Project Gutenberg. The file is cached locally so subsequent runs skip the download.

In [ ]:
CORPUS_DIR = Path("./corpus")
CORPUS_DIR.mkdir(exist_ok=True)
CORPUS_PATH = CORPUS_DIR / "sherlock_holmes.txt"

GUTENBERG_URL = "https://www.gutenberg.org/cache/epub/1661/pg1661.txt"

if not CORPUS_PATH.exists():
    print(f"Downloading from {GUTENBERG_URL}...")
    urllib.request.urlretrieve(GUTENBERG_URL, CORPUS_PATH)
    print("Download complete.")
else:
    print("Corpus already downloaded.")

corpus_text = CORPUS_PATH.read_text(encoding="utf-8")
print(f"Corpus size: {len(corpus_text):,} characters, ~{len(corpus_text.split()):,} words")

## Truncation Option

Indexing the full book takes several minutes and uses real API credits. Set `MAX_CHARS` to limit the text for a quicker demo, or set it to `None` to index everything.

In [ ]:
MAX_CHARS = 50_000  # Set to None for the full book

if MAX_CHARS is not None:
    text = corpus_text[:MAX_CHARS]
    print(f"Using first {MAX_CHARS:,} characters (~{len(text.split()):,} words)")
else:
    text = corpus_text
    print(f"Using full corpus: {len(text):,} characters")

## Initialize GraphRAG

We configure the instance with a domain description and entity types tailored to detective fiction.

In [ ]:
rag = GraphRAG(
    working_dir="./db/examples_book",
    domain=(
        "detective fiction: characters, locations, events, clues, "
        "and relationships in Sherlock Holmes stories"
    ),
    example_queries=(
        "Who is Sherlock Holmes?\n"
        "What happened in A Scandal in Bohemia?\n"
        "What is the relationship between Holmes and Watson?\n"
        "Which cases involve disguises or deception?"
    ),
    entity_types=["person", "location", "object", "event", "organization"],
)
print("GraphRAG initialized.")

## Index the Book

This step extracts entities and relationships from the text. For the full book, expect this to take several minutes.

In [ ]:
token_tracker.reset()
start = time.perf_counter()

num_entities, num_relations, num_chunks = rag.insert(text)

elapsed = time.perf_counter() - start
print(f"Indexing completed in {elapsed:.1f}s")
print(f"  Entities:  {num_entities}")
print(f"  Relations: {num_relations}")
print(f"  Chunks:    {num_chunks}")
print()
print(token_tracker.summary())

## Multi-Query Benchmark

Let's run a variety of question types — factual recall, analytical reasoning, and cross-story questions — and track latency and cost for each.

In [ ]:
QUESTIONS = [
    # Factual recall
    "Who is Irene Adler and what is her significance to Sherlock Holmes?",
    # Analytical
    "What methods does Holmes use to solve cases? Describe his investigative approach.",
    # Cross-story
    "What recurring themes or patterns appear across multiple Sherlock Holmes stories?",
    # Character relationship
    "Describe the relationship between Sherlock Holmes and Dr. Watson.",
]

results = []

for i, question in enumerate(QUESTIONS):
    token_tracker.reset()
    start = time.perf_counter()

    response = rag.query(question, params=QueryParam.balanced())

    elapsed = time.perf_counter() - start
    cost = token_tracker.total_cost

    results.append({
        "question": question,
        "answer": response.response,
        "latency": elapsed,
        "cost": cost,
        "num_entities": len(response.context.entities),
        "num_relations": len(response.context.relations),
        "num_chunks": len(response.context.chunks),
    })

    print(f"\nQ{i+1}: {question}")
    print(f"  Time: {elapsed:.1f}s | Cost: ${cost:.4f} | "
          f"Context: {len(response.context.entities)}E / "
          f"{len(response.context.relations)}R / "
          f"{len(response.context.chunks)}C")
    print(f"  Answer: {response.response[:200]}..." if len(response.response) > 200 else f"  Answer: {response.response}")

In [ ]:
# Summary table
print(f"{'#':<4} {'Latency':>8} {'Cost':>8} {'Entities':>9} {'Relations':>10} {'Chunks':>7}  Question")
print("-" * 100)
for i, r in enumerate(results):
    print(f"{i+1:<4} {r['latency']:>7.1f}s ${r['cost']:>6.4f} {r['num_entities']:>9} "
          f"{r['num_relations']:>10} {r['num_chunks']:>7}  {r['question'][:50]}")

total_latency = sum(r["latency"] for r in results)
total_cost = sum(r["cost"] for r in results)
print("-" * 100)
print(f"{'Total':<4} {total_latency:>7.1f}s ${total_cost:>6.4f}")

## Observations

Key takeaways from this notebook:

- **Indexing cost** scales with text length — the LLM processes each chunk for entity extraction.
- **Query latency** is relatively stable regardless of graph size, since retrieval uses efficient graph traversal + embedding similarity.
- **Cross-story questions** tend to retrieve more entities and relations, as the graph connects information across stories.
- Use `MAX_CHARS = None` to index the full book and see how the graph scales.

**Next steps:**
- [03_emails_and_chat.ipynb](03_emails_and_chat.ipynb) — Work with metadata-rich documents
- [06_graph_visualization.ipynb](06_graph_visualization.ipynb) — Export and visualize the knowledge graph